In [ ]:
# Use jax to compute
import jax
import jax.numpy as jnp
# using dataclass to store data
from dataclasses import dataclass

# pre-defined orbitals
orbital_list = ['s','p','d','f']

# using double precision
jax.config.update("jax_enable_x64", True)
default_type_complex = jnp.complex128
default_type_real = jnp.float64



# utilty

In [ ]:
#utilty ================================ BEGIN

# fermi-dirac funtion
def fermi_dirac(E,mu,kBT) -> float:
    return 1.0/(jnp.exp((E-mu)/kBT)+1)
# vamp it, E is an array, in  shape of (nE,), 
# return an F-D distrubution of a whole range of E
fermi_dirac_vamp = jax.vmap(fermi_dirac,in_axes=(0,None,None))

# calculate difference matrix
# only need to be computed once
def cal_center_matrix(center_list):
    '''
    param center_list: wannier center list, 2D array, shape (nwan,3)
    return: wannier center difference matrix, 3D array, shape (nwan,nwan,3)

    matrix[i,j] = vec[i] - vec[j]
    '''
    # check input
    assert center_list.ndim == 2, "center_list must be a 2D array"  
    assert center_list.shape[1] == 3, "center_list must have shape (nwan, 3)"
    # calculate wannier center difference matrix
    center_matrix = center_list[:, None, :] - center_list[None, :, :]
    return center_matrix

#utilty ================================ END

# data class

In [ ]:
# data class========================= BEGIN
@dataclass
class TB_model:
    HR: jnp.ndarray = None # (nR ,nwan,nwan)
    cell_center: jnp.ndarray = None # (nR,3)
    real_lattice: jnp.ndarray = None # (3,3)
    recip_lattice: jnp.ndarray = None # (3,3)
    center_list: jnp.ndarray = None # (nwan,3)
    AA_R: jnp.ndarray = None # (nR,nwan,nwan,3)

    # post init: calculate some useful quantities
    def __post_init__(self):
        self.nwan = self.HR.shape[-1] # number of wannier functions
        self.nR = self.HR.shape[0] # number of R points
        self.center_matrix = cal_center_matrix(self.center_list) # (nwan,nwan,3)

        b1 = self.recip_lattice[0]
        b2 = self.recip_lattice[1]
        b3 = self.recip_lattice[2]
        self.V = jnp.dot(b1,jnp.cross(b2,b3)) # volume of a reciprocal lattice cell


# data class========================= END

# IO functions

In [ ]:
# io ================================ BEGIN
def read_TB_model(file_name: str) -> TB_model:
    """
    Read a TB model from a npz file.

    Parameters
    ----------
    file_name : str
        The name of the npz or pickle file.

    Returns
    -------
    TB_model
        The TB model.
    """

    # check file extension
    if file_name.endswith(".npz"):
        data = jnp.load(file_name)
        HR = data['HR']
        cell_center = data['cRvec']
        center_list = data['Wan_center']
        AA_R = data['AA_R']
        real_lattice = data['real_lattice']
        recip_lattice = data['recip_lattice']

        
        return TB_model(HR=HR,
                        cell_center=cell_center,
                        center_list=center_list,
                        AA_R=AA_R,
                        real_lattice=real_lattice,
                        recip_lattice=recip_lattice)
    elif file_name.endswith(".pickle"):
        import pickle
        with open(file_name, "rb") as f:
            model = pickle.load(f)
        return model
    else:
        raise ValueError(f"Unknown file extension {file_name}")


def save_TB_model(model: TB_model, file_name: str):
    """
    Save a TB model to a pickle file.

    Parameters
    ----------
    model : TB_model
        The TB model to be saved.
    file_name : str
        The name of the pickle file.
    """
    import pickle
    with open(file_name, "wb") as f:
        pickle.dump(model, f)
# io ================================ END

# Operators

In [36]:
num_ao = { # num of atomic orbitals for each orbital type, considering spin
    's':2,
    'p':6,
    'd':10,
    'f':14, # no g orbital
}
# load pre-defined operator matrix
operator = jnp.load("operator.npz")

def build_operator_name_list(orbital_list:list[str]):
    """
    build operator name list for given orbital list

    param orbital_list: list of orbital names, e.g. ["spd","p"]
    return: operator name list repeat for each orbital type, 
    e.g. ["s","s","p","p","p","d","d","d"]
    """
    orbital_str = "".join(orbital_list)
    orbital_name_list = []
    for o in orbital_str:
        if o not in num_ao:
            raise ValueError(f"orbital type {o} not supported")
        num_repeat = num_ao[o]
        orbital_name_list.extend([o]*num_repeat)
 
    return orbital_name_list

def build_operator(orbital_list:list[str],use_LS_base:bool=False):
    """
    build operator matrix for given orbital list

    param orbital_list: list of orbital names, e.g. ["spd","p","sd",'f']
    param use_LS_base: whether to use LS coupled base, 
    default False, using real Atomic orbital basis
    return: S,L in 3D array, shape (norb,norb,3) and LS in 2D array, shape (norb,norb)
    """
    orbital_str = "".join(orbital_list)

    orbital_all = []
    for o in orbital_str:
        if o not in num_ao:
            raise ValueError(f"orbital type {o} not supported")
        orbital_all.append(num_ao[o])
    
    # init operator matrix
    norb = sum(orbital_all)
    S = jnp.zeros((norb,norb,3),dtype=default_type_complex)
    L = jnp.zeros((norb,norb,3),dtype=default_type_complex)

    # choose which operator matrix to use
    if use_LS_base:
        S_all = operator["S_LS"]
        L_all = operator["L_LS"]
    else:
        S_all = operator["S"]
        L_all = operator["L"]

    # seperate each orbitals into a dictionary
    S_AO = {
        2:S_all[0:2,0:2,:], # 2
        6:S_all[2:8,2:8,:], # 6
        10:S_all[8:18,8:18,:], # 10
        14:S_all[18:32,18:32,:], # 14    
    }
    L_AO = {
        2:L_all[0:2,0:2,:],# 2
        6:L_all[2:8,2:8,:],# 6
        10:L_all[8:18,8:18,:],# 10  
        14:L_all[18:32,18:32,:],# 14
    }  

    # fill L and S matrix
    i = 0
    for n_ao in orbital_all:
        start = i
        end = i + n_ao
        S = S.at[start:end,start:end,:].set(S_AO[n_ao])
        L = L.at[start:end,start:end,:].set(L_AO[n_ao])
        i = end
    
    # L dot S
    LS = L[:,:,0] @ S[:,:,0] + L[:,:,1] @ S[:,:,1] + L[:,:,2] @ S[:,:,2]

    return S,L,LS

    



In [40]:
res = build_operator(["sp","p","sd",'f'])
res2 = build_operator_name_list(["sp","sp","sd"])

# test

In [43]:
"".join(res2)

'ssppppppssppppppssdddddddddd'